# Medical Document Summarization — Full Pipeline on Colab

Runs **Phases 3, 4 and 5** of the project on a free Colab T4 GPU.

**Before you start**:
1. Click `Runtime > Change runtime type > Hardware accelerator: T4 GPU`.
2. Upload `pubmed_eval_sample.jsonl` to this session (left sidebar, folder icon, upload). 
   This is the 30-document file you built locally with `build_eval_sample.py`.
3. Run the cells in order. Total time on T4: ~25-40 minutes for all three approaches.

**What we do differently from local**:
- No Ollama. We use `transformers` directly with a quantized model that fits in T4's 16 GB VRAM.
- Everything else (TextRank, RAG retrieval, prompts) is identical to the local code.

## 0. Verify GPU is available

In [ ]:
!nvidia-smi

## 1. Install dependencies

Colab already has torch + transformers. We add what's missing.

In [ ]:
!pip install -q sentence-transformers chromadb rouge-score bert-score networkx==3.2.1
!pip install -q -U bitsandbytes accelerate
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Dependencies installed.')

## 2. Verify the eval sample is uploaded

If this errors, upload `pubmed_eval_sample.jsonl` from your local machine using the file browser on the left.

In [ ]:
import json, os

EVAL_PATH = 'pubmed_eval_sample.jsonl'
assert os.path.exists(EVAL_PATH), f'Upload {EVAL_PATH} first.'

with open(EVAL_PATH) as f:
    records = [json.loads(line) for line in f]
print(f'Loaded {len(records)} eval documents.')
print(f'First doc id: {records[0]["id"]}')
print(f'First article (first 300 chars): {records[0]["article"][:300]}')

## 3. Load the LLM (Mistral 7B, 4-bit quantized)

Quantization to 4 bits drops VRAM from ~14 GB to ~5 GB so we comfortably fit on T4. Quality loss is small for summarization.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()
print('Model loaded.')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 4. Generation helper

Drop-in replacement for the local `ollama.chat` call. Same signature, same determinism settings.

In [ ]:
import time

MAX_ARTICLE_WORDS = 4000

def truncate_article(article, max_words=MAX_ARTICLE_WORDS):
    words = article.split()
    if len(words) <= max_words:
        return article, False
    return ' '.join(words[:max_words]), True

def generate_summary(article, system_prompt, user_template, max_new_tokens=400, temperature=0.2):
    article_t, was_truncated = truncate_article(article)
    user_msg = user_template.format(article=article_t)
    messages = [
        {'role': 'user', 'content': system_prompt + '\n\n' + user_msg},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t0
    generated = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return {'text': generated, 'elapsed': elapsed, 'truncated': was_truncated}

# Smoke test
result = generate_summary(
    'Pneumonia is an infection that inflames the air sacs in one or both lungs. The air sacs may fill with fluid or pus.',
    'You are a medical assistant. Be concise.',
    'Summarize: {article}',
    max_new_tokens=100,
)
print(f'Generated in {result["elapsed"]:.1f}s:')
print(result['text'])

## 5. Phase 3 — TextRank baseline (fast, ~1 min on 30 docs)

In [ ]:
import numpy as np, networkx as nx, nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

def split_sentences(text, min_words=4):
    raw = nltk.sent_tokenize(text)
    merged = []
    for s in raw:
        s = s.strip()
        if not s: continue
        if merged and len(s.split()) < min_words:
            merged[-1] = merged[-1] + ' ' + s
        else:
            merged.append(s)
    return merged

def textrank_tfidf(text, num_sentences=7):
    sentences = split_sentences(text)
    if len(sentences) <= num_sentences:
        return ' '.join(sentences)
    vec = TfidfVectorizer(stop_words='english')
    tfidf = vec.fit_transform(sentences)
    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0.0)
    graph = nx.from_numpy_array(sim)
    scores = nx.pagerank(graph, max_iter=200, tol=1e-6)
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top = sorted(idx for idx, _ in ranked[:num_sentences])
    return ' '.join(sentences[i] for i in top)

preds_textrank = []
for rec in tqdm(records, desc='TextRank'):
    t0 = time.perf_counter()
    summary = textrank_tfidf(rec['article'])
    preds_textrank.append({
        'id': rec['id'],
        'approach': 'textrank_tfidf',
        'prediction': summary,
        'reference': rec['reference_summary'],
        'elapsed_seconds': round(time.perf_counter() - t0, 3),
    })

with open('predictions_textrank_tfidf.jsonl', 'w') as f:
    for p in preds_textrank:
        f.write(json.dumps(p) + '\n')
print(f'Done: {len(preds_textrank)} predictions saved.')

## 6. Phase 4 — LLM with structured prompt (no RAG)

Expected on T4: ~10-20 seconds per document, so ~5-10 min total.

In [ ]:
STRUCTURED_SYSTEM = (
    'You are a medical writing assistant. Your task is to summarize '
    'biomedical articles for clinicians.\n\n'
    'Strict rules:\n'
    '1. Base the summary ONLY on information present in the source article. '
    'Do NOT add facts, numbers, dosages, or conclusions that are not stated in the text.\n'
    '2. If a piece of information is not in the article, do not invent it.\n'
    '3. Preserve all quantitative details from the source (sample sizes, '
    'p-values, effect sizes, dosages) exactly as written.\n'
    '4. Use neutral, factual language. No speculation.'
)

STRUCTURED_USER = (
    'Summarize the following biomedical article. Use exactly these four sections, '
    'each one short paragraph:\n\nBackground:\nMethods:\nResults:\nConclusions:\n\n'
    'Total length: roughly 200 words.\n\nArticle:\n{article}\n\nStructured summary:'
)

preds_llm = []
for rec in tqdm(records, desc='LLM structured'):
    try:
        r = generate_summary(rec['article'], STRUCTURED_SYSTEM, STRUCTURED_USER, max_new_tokens=400)
        preds_llm.append({
            'id': rec['id'],
            'approach': 'llm_mistral_structured',
            'prediction': r['text'],
            'reference': rec['reference_summary'],
            'elapsed_seconds': round(r['elapsed'], 3),
            'input_truncated': r['truncated'],
        })
    except Exception as e:
        print(f'Error on {rec["id"]}: {e}')

with open('predictions_llm_mistral_structured.jsonl', 'w') as f:
    for p in preds_llm:
        f.write(json.dumps(p) + '\n')
avg = sum(p['elapsed_seconds'] for p in preds_llm) / len(preds_llm)
print(f'Done: {len(preds_llm)} predictions, avg {avg:.1f}s per doc.')

## 7. Phase 5 — LLM + RAG

### 7a. Build the knowledge base

In [ ]:
MEDICAL_KB = [
  {'term': 'creatinine', 'category': 'lab_value', 'content': 'Serum creatinine is a marker of kidney function. Normal range in adults: men 0.7-1.3 mg/dL, women 0.6-1.1 mg/dL.'},
  {'term': 'hemoglobin', 'category': 'lab_value', 'content': 'Hemoglobin measures oxygen-carrying capacity of red blood cells. Normal: men 13.5-17.5 g/dL, women 12.0-15.5 g/dL.'},
  {'term': 'blood pressure', 'category': 'vital_sign', 'content': 'Normal adult blood pressure: systolic <120 mmHg, diastolic <80 mmHg. Hypertension >=130/80 mmHg.'},
  {'term': 'HbA1c', 'category': 'lab_value', 'content': 'Glycated hemoglobin reflects average glucose over 2-3 months. Normal <5.7%. Diabetes >=6.5%.'},
  {'term': 'LDL cholesterol', 'category': 'lab_value', 'content': 'LDL cholesterol. Optimal <100 mg/dL. High 160-189. Very high >=190.'},
  {'term': 'troponin', 'category': 'lab_value', 'content': 'Cardiac troponin is the gold-standard marker of myocardial injury, central to diagnosis of acute MI.'},
  {'term': 'BNP', 'category': 'lab_value', 'content': 'B-type natriuretic peptide. Elevated in heart failure. <100 pg/mL makes acute HF unlikely; >400 strongly suggests it.'},
  {'term': 'CRP', 'category': 'lab_value', 'content': 'C-reactive protein, acute-phase reactant. Normal <10 mg/L. Used for inflammation and cardiovascular risk.'},
  {'term': 'myocardial infarction', 'category': 'disease', 'content': 'Acute MI is irreversible cardiac necrosis from ischemia. Diagnosed by troponin rise plus ECG/imaging evidence.'},
  {'term': 'stroke', 'category': 'disease', 'content': 'Acute neurological dysfunction from cerebral infarction (~87%) or hemorrhage. Time-critical for treatment.'},
  {'term': 'pneumonia', 'category': 'disease', 'content': 'Inflammation of lung parenchyma, usually infectious. Most common cause: Streptococcus pneumoniae.'},
  {'term': 'diabetes mellitus', 'category': 'disease', 'content': 'Chronic hyperglycemia. Type 1 autoimmune, Type 2 insulin resistance. Dx: HbA1c >=6.5%, fasting glucose >=126 mg/dL.'},
  {'term': 'heart failure', 'category': 'disease', 'content': 'Clinical syndrome of impaired cardiac output. HFrEF EF<=40%, HFmrEF 41-49%, HFpEF >=50%. NYHA classes I-IV.'},
  {'term': 'hypertension', 'category': 'disease', 'content': 'Sustained elevated BP. Major risk factor for stroke, MI, heart failure, CKD.'},
  {'term': 'COPD', 'category': 'disease', 'content': 'Progressive airflow limitation, usually from smoking. Spirometry: post-bronchodilator FEV1/FVC <0.70.'},
  {'term': 'statin', 'category': 'drug_class', 'content': 'HMG-CoA reductase inhibitors lower LDL. Examples: atorvastatin, rosuvastatin. Used for CV prevention.'},
  {'term': 'ACE inhibitor', 'category': 'drug_class', 'content': 'Block angiotensin I to II conversion. Examples: lisinopril, ramipril. Used in HTN, HF, CKD.'},
  {'term': 'beta-blocker', 'category': 'drug_class', 'content': 'Block beta-adrenergic receptors. Examples: metoprolol, bisoprolol. Used in HF, post-MI, HTN, arrhythmias.'},
  {'term': 'metformin', 'category': 'drug', 'content': 'First-line oral agent for type 2 diabetes. Reduces hepatic glucose. 500-2000 mg/day. GI side effects.'},
  {'term': 'aspirin', 'category': 'drug', 'content': 'Irreversible COX inhibitor. Low-dose (75-100 mg/day) for antiplatelet CV secondary prevention.'},
  {'term': 'warfarin', 'category': 'drug', 'content': 'Vitamin K antagonist oral anticoagulant. Monitor INR (target 2.0-3.0). Used in AF, mechanical valves, VTE.'},
  {'term': 'MRI', 'category': 'procedure', 'content': 'Magnetic resonance imaging. Strong soft tissue contrast, no ionizing radiation.'},
  {'term': 'CT scan', 'category': 'procedure', 'content': 'Computed tomography combines X-ray images. Fast, widely available, uses ionizing radiation.'},
  {'term': 'ECG', 'category': 'procedure', 'content': 'Electrocardiogram records cardiac electrical activity. 12-lead ECG essential for chest pain, arrhythmia, ischemia.'},
  {'term': 'PCR', 'category': 'procedure', 'content': 'Polymerase chain reaction amplifies DNA/RNA. Highly sensitive pathogen detection.'},
  {'term': 'randomized controlled trial', 'category': 'methodology', 'content': 'RCT randomly assigns participants to intervention vs control. Highest-quality single-study design for causal inference.'},
  {'term': 'meta-analysis', 'category': 'methodology', 'content': 'Statistical combination of multiple study results. Usually paired with a systematic review.'},
  {'term': 'p-value', 'category': 'methodology', 'content': 'Probability of data as extreme as observed if null is true. p<0.05 conventionally significant. Not an effect size.'},
  {'term': 'confidence interval', 'category': 'methodology', 'content': '95% CI contains the true parameter in 95% of repeated samples. Narrow CI = precise estimate.'},
  {'term': 'odds ratio', 'category': 'methodology', 'content': 'Compares odds of outcome between groups. OR=1 no effect, >1 higher, <1 lower odds.'},
  {'term': 'ICU', 'category': 'abbreviation', 'content': 'Intensive Care Unit. Hospital unit for life-threatening conditions needing monitoring and life support.'},
  {'term': 'ED', 'category': 'abbreviation', 'content': 'Emergency Department for urgent/life-threatening conditions.'},
  {'term': 'BMI', 'category': 'abbreviation', 'content': 'Body mass index = weight(kg)/height(m)^2. <18.5 underweight, 18.5-24.9 normal, 25-29.9 overweight, >=30 obese.'},
  {'term': 'GFR', 'category': 'abbreviation', 'content': 'Glomerular filtration rate. Normal >=90 mL/min/1.73m^2. CKD staged 1-5 by GFR.'},
  {'term': 'VTE', 'category': 'abbreviation', 'content': 'Venous thromboembolism: DVT and PE. Major preventable cause of in-hospital mortality.'},
  {'term': 'PCI', 'category': 'procedure', 'content': 'Percutaneous coronary intervention. Catheter-based coronary revascularization with stent placement.'},
]

import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()
try:
    client.delete_collection('medical_kb')
except Exception:
    pass
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
collection = client.create_collection('medical_kb', embedding_function=embed_fn, metadata={'hnsw:space': 'cosine'})
collection.add(
    ids=[f'kb_{i:03d}' for i in range(len(MEDICAL_KB))],
    documents=[f"{e['term']}. {e['content']}" for e in MEDICAL_KB],
    metadatas=[{'term': e['term'], 'category': e['category']} for e in MEDICAL_KB],
)
print(f'Indexed {len(MEDICAL_KB)} KB entries.')

### 7b. Run LLM + RAG on the eval sample

In [ ]:
RAG_SYSTEM = (
    'You are a medical writing assistant summarizing biomedical articles for clinicians.\n\n'
    'You receive two inputs:\n'
    '- ARTICLE: the source document. ALL summary content must come from it.\n'
    '- REFERENCE KNOWLEDGE: short factual notes on medical terms. Use ONLY to interpret terms and units. '
    'Do NOT add facts from REFERENCE KNOWLEDGE that are not in the ARTICLE.\n\n'
    'Strict rules:\n1. Base every claim on the ARTICLE.\n'
    '2. Do not introduce numbers, drugs, dosages, or findings absent from the ARTICLE.\n'
    '3. Preserve quantitative details from the ARTICLE exactly as written.\n'
    '4. Use neutral, factual language. No speculation.'
)

def retrieve_for_article(article, top_k=5):
    res = collection.query(query_texts=[article[:1500]], n_results=top_k)
    out = []
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        content = doc.split('. ', 1)[1] if '. ' in doc else doc
        out.append({'term': meta['term'], 'category': meta['category'], 'content': content, 'distance': float(dist)})
    return out

def format_retrieved(entries):
    return '\n'.join(f"- {e['term']} ({e['category']}): {e['content']}" for e in entries)

preds_rag = []
retrieval_log = []
for rec in tqdm(records, desc='LLM + RAG'):
    retrieved = retrieve_for_article(rec['article'], top_k=5)
    retrieval_log.append({'id': rec['id'], 'retrieved': [{'term': e['term'], 'distance': round(e['distance'], 4)} for e in retrieved]})
    rag_user = (
        'REFERENCE KNOWLEDGE (background only, do not add facts from here):\n'
        + format_retrieved(retrieved)
        + '\n\nARTICLE (source of truth):\n{article}\n\n'
        + 'Write a structured summary of the ARTICLE in four short paragraphs, '
        + 'approximately 200 words. Use these sections:\n\nBackground:\nMethods:\nResults:\nConclusions:\n\nStructured summary:'
    )
    try:
        r = generate_summary(rec['article'], RAG_SYSTEM, rag_user, max_new_tokens=400)
        preds_rag.append({
            'id': rec['id'],
            'approach': 'rag_mistral_top5',
            'prediction': r['text'],
            'reference': rec['reference_summary'],
            'elapsed_seconds': round(r['elapsed'], 3),
            'input_truncated': r['truncated'],
            'num_retrieved': len(retrieved),
        })
    except Exception as e:
        print(f'Error on {rec["id"]}: {e}')

with open('predictions_rag_mistral_top5.jsonl', 'w') as f:
    for p in preds_rag: f.write(json.dumps(p) + '\n')
with open('retrieval_log_rag_mistral_top5.jsonl', 'w') as f:
    for r in retrieval_log: f.write(json.dumps(r) + '\n')
print(f'Done: {len(preds_rag)} RAG predictions saved.')

## 8. Quick visual comparison (one example)

Pick a document and look at the three predictions side by side. This is your qualitative example for the report.

In [ ]:
DOC_ID = records[0]['id']  # change to any other id

def get_pred(preds, doc_id):
    for p in preds:
        if p['id'] == doc_id: return p['prediction']
    return '(not found)'

print(f'=== Document {DOC_ID} ===\n')
print('REFERENCE (gold):\n')
print(records[0]['reference_summary'][:1000])
print('\n--- TextRank ---\n')
print(get_pred(preds_textrank, DOC_ID)[:1000])
print('\n--- LLM (no RAG) ---\n')
print(get_pred(preds_llm, DOC_ID)[:1000])
print('\n--- LLM + RAG ---\n')
print(get_pred(preds_rag, DOC_ID)[:1000])

## 9. Download all predictions

These five files go back to your local project for Phase 6 (evaluation) and the final report.

In [ ]:
from google.colab import files
import os
for f in ['predictions_textrank_tfidf.jsonl',
          'predictions_llm_mistral_structured.jsonl',
          'predictions_rag_mistral_top5.jsonl',
          'retrieval_log_rag_mistral_top5.jsonl']:
    if os.path.exists(f):
        files.download(f)
    else:
        print(f'Missing: {f}')